# Tp03 : Étude de cas Yelp

## Création du Dataset

### Import des librairie

In [65]:
from pandas import read_csv, merge, DataFrame
from numpy import nan, floor
from utils import create_data_path, parse_hours

### Variable

In [66]:
export_path: str = "./restaurants_features.csv"
days: list[str] = ["lundi", "mardi", "mercredi", "jeudi", "vendredi", "samedi", "dimanche"]
data_set_path: dict[str, str] = {
    "avis"         : create_data_path("avis.csv"),
    "categories"   : create_data_path("categories.csv"),
    "checkin"      : create_data_path("checkin.csv"),
    "conseils"     : create_data_path("conseils.csv"),
    "horaires"     : create_data_path("horaires.csv"),
    "restaurants"  : create_data_path("restaurants.csv"),
    "services"     : create_data_path("services.csv"),
    "utilisateurs" : create_data_path("utilisateurs.csv")
}

c:\Users\DCS2003\Desktop\1-cegep\5-session05\5-collecte_interpretation\2-travaux_pratique\2-python\420-514-MV-TP03
c:\Users\DCS2003\Desktop\1-cegep\5-session05\5-collecte_interpretation\2-travaux_pratique\2-python\420-514-MV-TP03
c:\Users\DCS2003\Desktop\1-cegep\5-session05\5-collecte_interpretation\2-travaux_pratique\2-python\420-514-MV-TP03
c:\Users\DCS2003\Desktop\1-cegep\5-session05\5-collecte_interpretation\2-travaux_pratique\2-python\420-514-MV-TP03
c:\Users\DCS2003\Desktop\1-cegep\5-session05\5-collecte_interpretation\2-travaux_pratique\2-python\420-514-MV-TP03
c:\Users\DCS2003\Desktop\1-cegep\5-session05\5-collecte_interpretation\2-travaux_pratique\2-python\420-514-MV-TP03
c:\Users\DCS2003\Desktop\1-cegep\5-session05\5-collecte_interpretation\2-travaux_pratique\2-python\420-514-MV-TP03
c:\Users\DCS2003\Desktop\1-cegep\5-session05\5-collecte_interpretation\2-travaux_pratique\2-python\420-514-MV-TP03


### Charger Dataset

In [67]:
df_avis         = read_csv(data_set_path["avis"])
df_categories   = read_csv(data_set_path["categories"])
df_checking     = read_csv(data_set_path["checkin"])
df_conseils     = read_csv(data_set_path["conseils"])
df_horaires     = read_csv(data_set_path["horaires"])
df_restaurants  = read_csv(data_set_path["restaurants"])
df_services     = read_csv(data_set_path["services"])
df_utilisateurs = read_csv(data_set_path["utilisateurs"])

In [68]:
data = DataFrame(df_restaurants)
data = data.drop(["zone", "ferme"], axis = 1)

In [69]:
review_count = (df_avis
    .groupby("restaurant_id")
    .size()
    .reset_index(name = "review_count_total"))

In [70]:
positive_reviews = (df_avis[df_avis["etoiles"] >= 4]
    .groupby("restaurant_id")
    .size()
    .reset_index(name = "review_positive"))

In [71]:
review = merge(review_count, 
               positive_reviews, 
               left_on= "restaurant_id", 
               right_on="restaurant_id", 
               how="inner")

In [72]:
data = data.merge(review, 
             left_on  = "restaurant_id", 
             right_on = "restaurant_id", 
             how      = "inner")

In [73]:
data["review_count_total"] = (data["review_count_total"]
    .fillna(nan)
    .astype("int64"))

In [74]:
data["review_positive"] = (data["review_positive"]
    .fillna(nan)
    .astype("int64"))

In [75]:
data["positive_ratio"] = data["review_positive"] / data["review_count_total"]

In [76]:
checkins_total = (df_checking.groupby("restaurant_id")
    .size()
    .reset_index(name="checkins_total"))

In [77]:
data = data.merge(checkins_total, 
             left_on = "restaurant_id", 
             right_on = "restaurant_id", 
             how = "inner")

In [78]:
data["checkins_total"] = data["checkins_total"].fillna(nan).astype("int64")

In [79]:
name_counts = data["nom"].value_counts()

In [80]:
data["is_chain"] = data["nom"].map(lambda x: name_counts[x] >= 3)

In [81]:
prix_moyen = (
    df_services
    .groupby("restaurant_id")["prix"]
    .mean()
    .reset_index(name="prix_moyen"))

In [82]:
data = data.merge(prix_moyen, 
                  left_on = "restaurant_id", 
                  right_on = "restaurant_id", 
                  how = "left")

In [83]:
df_elite = df_utilisateurs[df_utilisateurs["elite"].notna()]

In [84]:
df_elite = df_utilisateurs[df_utilisateurs["elite"] != "[]"]

In [85]:
elite_id = df_elite[["utilisateur_id"]]

In [86]:
elite_id

,utilisateur_id
0,l6BmjZMeQD3rDxWUbiAiow
1,4XChL029mKr5hydo79Ljxg
2,bc8C_eETBWL0olvFSJJd0w
3,MM4RJAeH6yuaN8oZDSt0RA
4,TEtzbpgA2BFBrC0y0sCbfw
...,...
748242,qIKRzgQwcLsRbcjFYD1jkg
748243,vyK8txmse-HVk7hAKCvuMQ
748244,gCfFBY-cO1Y0TxxKX89-ig
748245,Z2uEg8DIk1twiNC6NV7e9w


In [87]:
elite_reviews = df_avis.merge(elite_id,
    on="utilisateur_id",
    how="inner")


In [88]:
elite_users_count = (elite_reviews
                     .groupby("restaurant_id")["utilisateur_id"]
                     .nunique()
                     .reset_index(name="elite_users_count"))

In [89]:
data = data.merge(elite_users_count,
    on="restaurant_id",
    how="left")

In [90]:
for d in days:
    df_horaires[d] = df_horaires[d].apply(parse_hours)

In [91]:
df_horaires["avg_open_hours"] = df_horaires[days].mean(axis=1)

In [92]:
df_horaires["avg_open_hours"] = floor(df_horaires["avg_open_hours"] * 100) / 100

In [93]:
data = data.merge(df_horaires[["restaurant_id", "avg_open_hours"]],
    on="restaurant_id",
    how="left")

In [94]:
data

,restaurant_id,nom,moyenne_etoiles,ville,review_count_total,review_positive,positive_ratio,checkins_total,is_chain,prix_moyen,elite_users_count,avg_open_hours
0,lCwqJWMxvIUQt1Re_tDn4w,Denny's,2.5,Las Vegas,72,22,0.305556,181,True,2.0,70,0.00
1,pd0v6sOqpLhFJ7mkpIaixw,Ike's Love & Sandwiches,4.0,Phoenix,108,82,0.759259,492,False,2.0,105,9.14
2,0vhi__HtC2L4-vScgDFdFw,Midori Japanese Cafe,3.5,Calgary,49,33,0.673469,157,False,2.0,48,9.64
3,t65yfB9v9fqlhAkLnnUXdg,Pho U,3.5,Toronto,36,21,0.583333,18,False,1.0,35,8.14
4,i7_JPit-2kAbtRTLkic2jA,John & Sons Oyster House,4.0,Toronto,88,63,0.715909,110,False,3.0,86,7.92
...,...,...,...,...,...,...,...,...,...,...,...,...
31835,cjZfgcQwA6KmQ_ANWKN2aw,Bruegger's Bagels,3.5,McMurray,6,4,0.666667,37,True,1.0,6,8.00
31836,Hq2edcOTjse7wjK2CwBijQ,Bistro Pointe-Claire,3.5,Pointe-Claire,11,6,0.545455,5,False,2.0,11,9.78
31837,7KlpgRjjAmVabPzxcExs0g,Taco Mex,4.0,Phoenix,11,8,0.727273,32,False,1.0,11,0.00
31838,0fY-zYyP2fDmp2YXFsuNTg,Gotham Provisions Company,4.0,Sun Prairie,18,13,0.722222,7,False,1.0,18,5.00
